In [25]:
from ingest import load_faq_data

documents = load_faq_data()

In [26]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)
        

161

In [27]:
documents = documents_llm

In [38]:
doc = documents[0]
print(doc['id'])
print(doc['course'])
print(doc['section'])
print(doc['question'])  
print(doc['answer'])

74eb249bbf
llm-zoomcamp
General Course-Related Questions
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [28]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [39]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [35]:
from dotenv import load_dotenv
from openai import OpenAI
import os


load_dotenv(dotenv_path="/workspaces/ramiro-llm-zoomcamp/.env")
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

openai_client = OpenAI()

In [40]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [41]:
import json

user_prompt = json.dumps(doc)

In [42]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [14]:
print(type(user_prompt))
print(user_prompt)

<class 'str'>
{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."}


In [43]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [44]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [50]:
response.output_parsed.questions

['Is it too late to join the course if I found it after it started?',
 'Can I still enroll even though I just discovered this course?',
 'If I join late, do I still have a chance to get a certificate?',
 'What do I need to do to get the certificate if I enter the course now?',
 'Are project submissions still open for someone who just joined the course?']

In [45]:
len(documents)

161

In [46]:
from evaluation_utils import llm_structured
from evaluation_utils import calc_price

In [51]:
result, answer_usage= llm_structured(
    openai_client, 
    data_gen_instructions,
    user_prompt,
    Questions
)

In [52]:
answer_usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0, cache_write_tokens_details={'ttl_12h_tokens': 0, 'ttl_30m_tokens': 0}), output_tokens=82, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=289)

In [53]:
calc_price(answer_usage)

{'input_cost': 0.00015525, 'output_cost': 0.000369, 'total_cost': 0.00052425}

In [54]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still join the course if I just found out about it?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start the course now?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to qualify for a certificate if I started late?',
  'document': '74eb249bbf'},
 {'question': 'Do I have to submit the project before submissions close to get certified?',
  'document': '74eb249bbf'}]

In [55]:
import pandas as pd

In [56]:
pd.DataFrame(records)

,question,document
0,Can I still join the course if I just found ou...,74eb249bbf
1,Is it too late to start the course now?,74eb249bbf
2,"If I join late, can I still get a certificate?",74eb249bbf
3,What do I need to do to qualify for a certific...,74eb249bbf
4,Do I have to submit the project before submiss...,74eb249bbf
